# Load Data

In [1]:
# Import required libraries
import mne
import numpy as np
import os
from pathlib import Path

# Suppress MNE warnings for cleaner output
mne.set_log_level('WARNING')

In [2]:
# Set dataset path
dataset_path = Path('/home/aloo/CNS_Summer_Project_1/Infants_data')

# Get list of all subjects
subjects = sorted([d for d in os.listdir(dataset_path) if d.startswith('sub-NORB')])
print(f"Found {len(subjects)} subjects")

Found 103 subjects


In [3]:
# Function to find EDF files for a subject
def find_edf_files(subject_id):
    """Find all EDF files for a given subject."""
    subject_path = dataset_path / subject_id
    edf_files = []
    
    if subject_path.exists():
        # Search for EDF files in subject directory
        edf_files = list(subject_path.rglob('*.edf'))
    
    return edf_files

# Test with first subject
test_subject = subjects[63]
edf_files = find_edf_files(test_subject)
print(f"\n{test_subject}:")
print(f"  Found {len(edf_files)} EDF file(s)")
for edf in edf_files:
    print(f"  - {edf.name}")


sub-NORB00064:
  Found 3 EDF file(s)
  - sub-NORB00064_ses-2_task-EEG_eeg.edf
  - sub-NORB00064_ses-1_task-EEG_eeg.edf
  - sub-NORB00064_ses-3_task-EEG_eeg.edf


In [4]:
# Function to load EDF file
def load_edf_data(edf_path):
    """
    Load EDF file with MNE and extract EEG channels.
    
    Parameters:
    -----------
    edf_path : str or Path
        Path to the EDF file
        
    Returns:
    --------
    raw : mne.io.Raw
        Raw EEG data
    """
    # Load the EDF file
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    
    # Get info
    n_channels = len(raw.ch_names)
    sampling_rate = raw.info['sfreq']
    duration = raw.times[-1]
    
    print(f"\nLoaded: {Path(edf_path).name}")
    print(f"  Channels: {n_channels}")
    print(f"  Sampling rate: {sampling_rate} Hz")
    print(f"  Duration: {duration:.2f} seconds")
    print(f"  Channel names: {raw.ch_names}")
    
    return raw

# Load the first EDF file as a test
if edf_files:
    raw = load_edf_data(edf_files[0])
else:
    print("No EDF files found for testing")


Loaded: sub-NORB00064_ses-2_task-EEG_eeg.edf
  Channels: 19
  Sampling rate: 200.0 Hz
  Duration: 360.00 seconds
  Channel names: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ']


In [5]:
# Extract EEG data as numpy array
if 'raw' in locals():
    # Get the data
    data = raw.get_data()  # Shape: (n_channels, n_timepoints)
    sampling_rate = raw.info['sfreq']
    channel_names = raw.ch_names
    
    print(f"\nExtracted data shape: {data.shape}")
    print(f"  Number of channels: {data.shape[0]}")
    print(f"  Number of timepoints: {data.shape[1]}")
    print(f"  Duration: {data.shape[1] / sampling_rate:.2f} seconds")


Extracted data shape: (19, 72000)
  Number of channels: 19
  Number of timepoints: 72000
  Duration: 360.00 seconds


# Preprocess

In [6]:
# Detect and remove bad channels
def detect_bad_channels(raw, flat_threshold=1e-12, noise_threshold=1e-3):
    """
    Detect bad channels (flat or extremely noisy).
    
    Parameters:
    -----------
    raw : mne.io.Raw
        Raw EEG data
    flat_threshold : float
        Threshold for flat channels (std < threshold)
    noise_threshold : float
        Threshold for noisy channels (std > threshold)
        
    Returns:
    --------
    bad_channels : list
        List of bad channel names
    """
    data = raw.get_data()
    channel_names = raw.ch_names
    bad_channels = []
    
    # Define reference/ground channel names to exclude
    ref_ground_channels = {'PG1', 'PG2', 'REF', 'GND', 'GROUND', '25+', '26+', '27+'}
    
    for i, ch_name in enumerate(channel_names):
        ch_data = data[i, :]
        ch_std = np.std(ch_data)
        ch_range = np.max(ch_data) - np.min(ch_data)
        
        is_bad = False
        
        # Check if flat
        if ch_std < flat_threshold:
            is_bad = True
        
        # Check if too noisy
        elif ch_std > noise_threshold:
            is_bad = True
        
        # Check if constant value
        elif ch_range == 0:
            is_bad = True
        
        # Check for reference/ground channels
        elif ch_name.upper() in ref_ground_channels:
            is_bad = True
        
        if is_bad:
            bad_channels.append(ch_name)
    
    print(f"\nFound {len(bad_channels)} bad channel(s): {bad_channels}")
    
    return bad_channels

# Detect bad channels
if 'raw' in locals():
    bad_channels = detect_bad_channels(raw)
    
    # Remove bad channels if any found
    if bad_channels:
        raw_clean = raw.copy()
        raw_clean.drop_channels(bad_channels)
        print(f"\nRemoved {len(bad_channels)} bad channel(s)")
        print(f"Remaining channels ({len(raw_clean.ch_names)}): {raw_clean.ch_names}")
    else:
        raw_clean = raw.copy()
        print("\nNo bad channels detected. Proceeding with all channels.")
    
    # Create raw_filtered alias (data is already filtered at acquisition)
    raw_filtered = raw_clean  # For compatibility with downstream code
else:
    print("Error: 'raw' data not found. Please run Step 1 first.")


Found 0 bad channel(s): []

No bad channels detected. Proceeding with all channels.


In [7]:
# Segment data based on annotation file
import pandas as pd

def load_annotations(subject_id, session_id, dataset_path):
    """
    Load annotation file for a given subject and session.
    Handles TSV files with trailing tabs/whitespace robustly.
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-NORB00001')
    session_id : str
        Session ID (e.g., 'ses-1')
    dataset_path : Path
        Path to dataset directory
        
    Returns:
    --------
    annotations_df : pd.DataFrame or None
        DataFrame with columns: onset, duration, label
    """
    annotations_path = dataset_path / 'derivatives' / 'NeuronicEEG' / subject_id / session_id / 'eeg' / f'{subject_id}_{session_id}_task-EEG_annotations.tsv'
    
    if annotations_path.exists():
        # Read file line by line to handle trailing tabs properly
        with open(annotations_path, 'r') as f:
            lines = f.readlines()
        
        # Parse manually to avoid pandas issues with trailing tabs
        data = []
        header = None
        
        for i, line in enumerate(lines):
            # Split by tab and strip whitespace from each field
            fields = [field.strip() for field in line.strip().split('\t')]
            
            if i == 0:
                # Header row
                header = fields[:3]  # Only take first 3 columns: onset, duration, label
            else:
                # Data row - only take first 3 fields
                if len(fields) >= 3 and fields[0] and fields[1]:  # Ensure we have onset and duration
                    try:
                        onset = float(fields[0])
                        duration = float(fields[1])
                        label = fields[2] if fields[2] else 'eyes_closed'  # Default if empty
                        data.append([onset, duration, label])
                    except ValueError:
                        continue  # Skip malformed lines
        
        if data:
            annotations_df = pd.DataFrame(data, columns=['onset', 'duration', 'label'])
            return annotations_df
        else:
            print(f"  Warning: No valid data in annotations file: {annotations_path}")
            return None
    else:
        print(f"  Warning: No annotations file found at {annotations_path}")
        return None

def segment_data_from_annotations(raw, annotations_df, label='eyes_closed'):
    """
    Segment EEG data based on annotations file.
    Only extracts segments marked with the specified label (e.g., 'eyes_closed').
    
    Parameters:
    -----------
    raw : mne.io.Raw
        Raw EEG data
    annotations_df : pd.DataFrame
        DataFrame with columns: onset, duration, label
    label : str
        Label to filter segments by (default: 'eyes_closed')
        
    Returns:
    --------
    segments : list of numpy arrays
        List of data segments, each with shape (n_channels, n_samples_per_segment)
    segment_info : list of dict
        List of dictionaries containing segment metadata (onset, duration)
    """
    sfreq = raw.info['sfreq']
    data = raw.get_data()
    n_channels, n_samples = data.shape
    
    # Filter annotations by label (case-insensitive + Spanish support)
    # Accept both 'eyes_closed' and Spanish 'ojos_cerrados'
    label_lower = label.lower()
    filtered_annotations = annotations_df[
        (annotations_df['label'].str.lower() == label_lower) |
        (annotations_df['label'].str.lower() == 'ojos_cerrados') |
        (annotations_df['label'].str.lower() == 'vigilia_ojos_abiertos') |
        (annotations_df['label'].str.lower() == 'sue o_espontaneo')
    ]
    
    segments = []
    segment_info = []
    
    print(f"\nSegmenting data from annotations file...")
    print(f"  Total annotations: {len(annotations_df)}")
    print(f"  Unique labels found: {annotations_df['label'].unique().tolist()}")
    print(f"  Annotations matching '{label}' {len(filtered_annotations)}")
    
    for idx, row in filtered_annotations.iterrows():
        onset = row['onset']  # in seconds
        duration = row['duration']  # in seconds
        
        # Convert to sample indices
        start_sample = int(onset * sfreq)
        end_sample = int((onset + duration) * sfreq)
        
        # Check if segment is within bounds
        if start_sample >= 0 and end_sample <= n_samples:
            segment = data[:, start_sample:end_sample]
            segments.append(segment)
            segment_info.append({
                'onset': onset,
                'duration': duration,
                'start_sample': start_sample,
                'end_sample': end_sample
            })
        else:
            print(f"  Warning: Segment at onset={onset}s exceeds data bounds, skipping")
    
    print(f"\nExtracted {len(segments)} valid segments")
    
    return segments, segment_info

# Apply segmentation based on annotations
if 'raw_clean' in locals():
    # Extract subject and session info from the loaded EDF file
    if 'edf_files' in locals() and edf_files:
        edf_name = edf_files[0].stem
        parts = edf_name.split('_')
        subject_id = parts[0]  # e.g., 'sub-NORB00042'
        session_id = parts[1] if len(parts) > 1 and 'ses' in parts[1] else 'ses-1'
        
        # Load annotations
        annotations_df = load_annotations(subject_id, session_id, dataset_path)
        
        if annotations_df is not None:
            # Segment data based on annotations
            segments, segment_info = segment_data_from_annotations(raw_clean, annotations_df, label='eyes_closed')
            
            if len(segments) > 0:
                print(f"\n✓ Successfully segmented data for {subject_id} {session_id}")
            else:
                print(f"\n⚠ No valid segments found for {subject_id} {session_id}")
        else:
            print(f"\n⚠ Could not load annotations. Skipping segmentation.")
    else:
        print("Error: EDF file information not found.")
else:
    print("Error: 'raw_clean' data not found. Please run bad channel detection first.")


Segmenting data from annotations file...
  Total annotations: 20
  Unique labels found: ['eyes_closed']
  Annotations matching 'eyes_closed' 20

Extracted 20 valid segments

✓ Successfully segmented data for sub-NORB00064 ses-2


In [9]:
# Function to calculate Pearson correlation matrices for segmented EEG epochs
def calculate_epoch_correlations(segments):
    """
    Calculate Pearson correlation matrix for each segmented EEG epoch.
    Parameters:
        segments (list of np.ndarray): List of EEG segments, each of shape (n_channels, n_samples_per_segment)
    Returns:
        correlation_matrices (list of np.ndarray): List of correlation matrices (n_channels x n_channels)
    """
    correlation_matrices = []
    for i, segment in enumerate(segments):
        corr_matrix = np.corrcoef(segment)
        correlation_matrices.append(corr_matrix)
        print(f"Epoch {i+1}: Correlation matrix shape: {corr_matrix.shape}")
    print(f"\nCalculated Pearson correlation matrices for {len(correlation_matrices)} epochs.")
    return correlation_matrices

# Example usage
if 'segments' in locals() and segments:
    correlation_matrices = calculate_epoch_correlations(segments)
else:
    print("No segmented data found. Please run segmentation first.")

Epoch 1: Correlation matrix shape: (19, 19)
Epoch 2: Correlation matrix shape: (19, 19)
Epoch 3: Correlation matrix shape: (19, 19)
Epoch 4: Correlation matrix shape: (19, 19)
Epoch 5: Correlation matrix shape: (19, 19)
Epoch 6: Correlation matrix shape: (19, 19)
Epoch 7: Correlation matrix shape: (19, 19)
Epoch 8: Correlation matrix shape: (19, 19)
Epoch 9: Correlation matrix shape: (19, 19)
Epoch 10: Correlation matrix shape: (19, 19)
Epoch 11: Correlation matrix shape: (19, 19)
Epoch 12: Correlation matrix shape: (19, 19)
Epoch 13: Correlation matrix shape: (19, 19)
Epoch 14: Correlation matrix shape: (19, 19)
Epoch 15: Correlation matrix shape: (19, 19)
Epoch 16: Correlation matrix shape: (19, 19)
Epoch 17: Correlation matrix shape: (19, 19)
Epoch 18: Correlation matrix shape: (19, 19)
Epoch 19: Correlation matrix shape: (19, 19)
Epoch 20: Correlation matrix shape: (19, 19)

Calculated Pearson correlation matrices for 20 epochs.


In [10]:
# Function to average a list of correlation matrices and return the absolute value

def average_abs_correlation_matrices(correlation_matrices):
    """
    Average a list of correlation matrices into a single matrix and return the absolute value.
    Parameters:
        correlation_matrices (list of np.ndarray): List of correlation matrices (n_channels x n_channels)
    Returns:
        avg_abs_correlation_matrix (np.ndarray): Averaged absolute correlation matrix (n_channels x n_channels)
    """
    stacked = np.stack(correlation_matrices, axis=0)
    avg_correlation_matrix = np.mean(stacked, axis=0)
    avg_abs_correlation_matrix = np.abs(avg_correlation_matrix)
    print(f"Averaged absolute correlation matrix shape: {avg_abs_correlation_matrix.shape}")
    return avg_abs_correlation_matrix

# Example usage
if 'correlation_matrices' in locals() and correlation_matrices:
    avg_abs_correlation_matrix = average_abs_correlation_matrices(correlation_matrices)
else:
    print("No correlation matrices found. Please run the correlation calculation first.")

Averaged absolute correlation matrix shape: (19, 19)


In [11]:
# Function to invert the averaged absolute correlation matrix (1 - |r|)
def invert_correlation_matrix(abs_corr_matrix):
    """
    Invert the absolute correlation matrix to obtain weights: 1 - |r|.
    Parameters:
        abs_corr_matrix (np.ndarray): Averaged absolute correlation matrix (n_channels x n_channels)
    Returns:
        inverted_matrix (np.ndarray): Inverted weight matrix (n_channels x n_channels)
    """
    inverted_matrix = 1.0 - abs_corr_matrix
    return inverted_matrix

# Example usage
if 'avg_abs_correlation_matrix' in locals():
    inverted_matrix = invert_correlation_matrix(avg_abs_correlation_matrix)
    print(f"Inverted matrix shape: {inverted_matrix.shape}")
else:
    print("No averaged absolute correlation matrix found. Please run the previous step first.")

Inverted matrix shape: (19, 19)


In [12]:
# Function to calculate betweenness centrality from a weighted adjacency matrix
import networkx as nx

def calculate_betweenness_centrality(weight_matrix, channel_names=None, normalized=True):
    """
    Calculate betweenness centrality for a weighted undirected network.
    Parameters:
        weight_matrix (np.ndarray): Weighted adjacency matrix (n_channels x n_channels)
        channel_names (list, optional): List of channel names for node labels
        normalized (bool): Whether to normalize betweenness centrality (default: True)
    Returns:
        centrality_dict (dict): Mapping from node label to betweenness centrality
    """
    n = weight_matrix.shape[0]
    if channel_names is None:
        channel_names = list(range(n))
    G = nx.Graph()
    # Add nodes
    for i in range(n):
        G.add_node(channel_names[i])
    # Add weighted edges (upper triangle, no self-loops)
    for i in range(n):
        for j in range(i+1, n):
            weight = weight_matrix[i, j]
            if not np.isnan(weight):
                G.add_edge(channel_names[i], channel_names[j], weight=weight)
    # Compute betweenness centrality (using edge weights as distances)
    centrality = nx.betweenness_centrality(G, weight='weight', normalized=normalized)
    return centrality

# Example usage
if 'inverted_matrix' in locals():
    # Use channel_names if available
    if 'channel_names' in locals():
        centrality = calculate_betweenness_centrality(inverted_matrix, channel_names=channel_names)
    else:
        centrality = calculate_betweenness_centrality(inverted_matrix)
    print("Betweenness centrality:")
    print(centrality)
else:
    print("No inverted matrix found. Please run the previous step first.")

Betweenness centrality:
{'Fp1': 0.0, 'Fp2': 0.0, 'F3': 0.006535947712418301, 'F4': 0.0196078431372549, 'C3': 0.0, 'C4': 0.0, 'P3': 0.0196078431372549, 'P4': 0.026143790849673203, 'O1': 0.0, 'O2': 0.0, 'F7': 0.0, 'F8': 0.0, 'T3': 0.0, 'T4': 0.0, 'T5': 0.0, 'T6': 0.0, 'FZ': 0.006535947712418301, 'CZ': 0.0, 'PZ': 0.0}
